> **Validation status:** historical metrics are unverified after the SES and moving-average tuning corrections. Supply the original Yahoo Finance CSV and rerun the corrected scripts. This notebook is an archived companion, not evidence of a new execution.

# AAPL Stock Forecasting Analysis

This project evaluates simple forecasting methods for AAPL daily closing prices.

## Evaluation design
- **Training:** January of each year
- **Test:** February–December of the same year
- **Metrics:** Mean Absolute Deviation (MAD) and Mean Absolute Percentage Error (MAPE)
- **Models:** Naive, Moving Average, Simple Exponential Smoothing, Linear Trend, Trend + Seasonal (21 trading-day cycle)

All trend and seasonal parameters are estimated **using January training data only**. No test-period observations are used to fit the forecasts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

DATA = Path("../data/processed")
RESULTS = Path("../results")
RESULTS.mkdir(parents=True, exist_ok=True)

def mad(actual, forecast):
    return np.mean(np.abs(np.asarray(actual) - np.asarray(forecast)))

def mape(actual, forecast):
    actual, forecast = np.asarray(actual), np.asarray(forecast)
    return np.mean(np.abs((actual - forecast) / actual)) * 100

def naive_forecast(train, horizon):
    return np.repeat(train[-1], horizon)

def moving_average_forecast(train, horizon, window):
    return np.repeat(np.mean(train[-window:]), horizon)

def ses_forecast(train, horizon, alpha):
    level = train[0]
    for value in train[1:]:
        level = alpha * value + (1-alpha) * level
    return np.repeat(level, horizon)

def choose_ma_window(train):
    candidates = range(2, min(10, len(train)))
    return min(candidates, key=lambda n: mad(train[n:], [np.mean(train[i-n:i]) for i in range(n, len(train))]))

def choose_ses_alpha(train):
    candidates = np.linspace(0.1, 0.9, 9)
    def error(alpha):
        level = train[0]
        errors = []
        for value in train[1:]:
            level = alpha * value + (1-alpha) * level
            errors.append(abs(value-level))
        return np.mean(errors)
    return min(candidates, key=error)

def trend_season_forecast(train, horizon, cycle=21):
    x = np.arange(len(train))
    slope, intercept, *_ = stats.linregress(x, train)
    trend = intercept + slope*x
    residuals = train - trend
    seasonal = np.array([residuals[i::cycle].mean() if len(residuals[i::cycle]) else 0 for i in range(cycle)])
    seasonal -= seasonal.mean()
    future_x = np.arange(len(train), len(train)+horizon)
    future_trend = intercept + slope*future_x
    future_season = seasonal[future_x % cycle]
    return future_trend, future_trend + future_season

In [ ]:
def analyze_year(year):
    df = pd.read_csv(DATA / f"AAPL_{year}_cleaned.csv", parse_dates=["Date"])
    train = df[df.Date.dt.month == 1]
    test = df[df.Date.dt.month > 1]
    y_train = train["Price"].to_numpy()
    y_test = test["Price"].to_numpy()
    h = len(y_test)

    ma_window = choose_ma_window(y_train)
    alpha = choose_ses_alpha(y_train)
    trend, trend_season = trend_season_forecast(y_train, h)

    forecasts = {
        "Naive": naive_forecast(y_train, h),
        f"MA (n={ma_window})": moving_average_forecast(y_train, h, ma_window),
        f"SES (alpha={alpha:.1f})": ses_forecast(y_train, h, alpha),
        "Linear Trend": trend,
        "Trend + Seasonal (21-day cycle)": trend_season
    }

    rows = []
    for method, forecast in forecasts.items():
        rows.append({"Year": year, "Method": method, "MAD": mad(y_test, forecast), "MAPE": mape(y_test, forecast)})

    results = pd.DataFrame(rows).sort_values("MAD")
    display(results)

    plt.figure(figsize=(12,6))
    plt.plot(train.Date, y_train, label="January training")
    plt.plot(test.Date, y_test, label="Actual Feb-Dec")
    for method, forecast in forecasts.items():
        plt.plot(test.Date, forecast, linewidth=1.3, label=method)
    plt.title(f"AAPL {year}: Out-of-sample forecasts")
    plt.xlabel("Date"); plt.ylabel("Closing price")
    plt.legend(fontsize=8); plt.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
    return results

results_2023 = analyze_year(2023)
results_2024 = analyze_year(2024)

In [ ]:
comparison = results_2023.merge(results_2024, on="Method", suffixes=("_2023", "_2024"))
comparison["MAD change"] = comparison["MAD_2024"] - comparison["MAD_2023"]
comparison["MAPE change"] = comparison["MAPE_2024"] - comparison["MAPE_2023"]
comparison.sort_values("MAD_2024")

## Interpretation

The corrected out-of-sample setup shows that the simple level methods are more competitive than the trend-based methods when only January is available for training.

This is an important result rather than a failure: a model should not be presented as superior simply because it is more sophisticated. The evaluation is based on held-out February–December observations.

### Reference-class adjustment

The original project included a reference-class adjustment based on peer companies. The available project files do **not** contain the required external peer datasets, and the previous implementation used synthetic peer series. That approach is not retained in the portfolio version because it would not support a defensible empirical claim.

The peer adjustment is therefore excluded rather than presented as if it were based on real comparable-company data.